# Correcting P-vals for MWAS findings for *totiviridiae* family
### Author: Oscar Heath
### Date: Sept 12 2026

The file results.parquet contains 80K MWAS results for *totiviridiae*, with associated p-values. These raw p-values have not undergone multiple test correction. We use the Benjamin-Hochberg multiple test correction method to compute corrected p-values, with $\alpha = 0.05$, and save all significant findings.

In [11]:
import polars as pl
from statsmodels.stats.multitest import multipletests

In [13]:
df = pl.read_parquet("../data/results.parquet")
significant_results = df.filter(pl.col("pvalue") < 0.05)

In [14]:
pvals = df.select(pl.col("pvalue")).to_numpy().flatten()
rejected, corrected_pvals, _, _ = multipletests(pvals, alpha=0.05, method='fdr_bh')

In [15]:
# add a column to the dataframe with the corrected p-values
df = df.with_columns(pl.Series("corrected_pvalue", corrected_pvals))

# get the significant results
significant_results = df.filter(pl.col("corrected_pvalue") < 0.05)

In [16]:
significant_results.write_parquet("../data/significant_results.parquet")

In [17]:
significant_results

bioproject,col_name,col_value,statistic,pvalue,mean_positive,std_positive,count_positive,mean_negative,std_negative,count_negative,corrected_pvalue
str,list[str],list[str],f64,f64,f64,f64,i32,f64,f64,i32,f64
"""PRJNA448599""","[""source_name_attrib"", ""phenotype_attrib""]","[""Single EHT cells"", ""Cdh5+ Gfi1+ Gfi1b- c-kit+""]",56.0,0.000311,0.705771,0.421236,7,0.13049,0.052028,8,0.035885
"""PRJNA693645""","[""env_broad_scale_attrib""]","[""Lung""]",2154.0,0.000293,0.240966,0.163286,14,78.500284,545.170104,707,0.034224
"""PRJNA693645""","[""env_broad_scale_attrib""]","[""Intestines""]",7681.0,0.000056,1.255677,3.048308,37,81.076921,554.088585,684,0.009219
"""PRJNA487968""","[""strain_attrib""]","[""S288C""]",273.0,0.000005,7.274563,6.158639,24,25.923064,21.652283,63,0.001186
"""PRJNA812896""","[""manuscript_sample_name_attrib""]","[""Donor 6 post""]",8.0,0.000076,0.026548,0.013135,7,1.340764,2.085031,42,0.011857
…,…,…,…,…,…,…,…,…,…,…,…
"""PRJEB44083""","[""collection_date_attrib""]","[""2019-09-10""]",0.0,0.000119,0.063964,0.000597,2,0.207691,0.136583,182,0.017036
"""PRJNA352533""","[""stress_attrib""]","[""HeatShock""]",17336.0,1.7423e-10,1719.35609,646.753128,84,1298.19392,1188.019116,283,1.4158e-7
"""PRJNA352533""","[""stress_attrib""]","[""NaCl""]",6436.0,1.7423e-10,1298.19392,1188.019116,283,1719.35609,646.753128,84,1.4158e-7
